In [1]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '../examples')

import warnings
warnings.filterwarnings('ignore')

from sldata.behavior_utils import get_file_paths
from kalman_filter_decoder import run_kalman_filter

In [2]:
# --- Dataset parameters ---
base_path  = 'd:/data/'
experiment = 'clickbait-motivate'

# Sessions to skip (index into event_paths after initial discovery)
# e.g. {48: '7012-m15 incomplete'}
skip_indices = {48}

# --- Decoder parameters ---
decoder_params = dict(
    bin_size_ms             = 30.0,
    test_fraction           = 0.2,
    neural_smooth_sigma     = 0.0,
    pos_smooth_sigma        = 0.0,
    n_permutations          = 1000,
    ridge_alpha             = 1.0,
    obs_noise_scale         = 1.0,
    mahal_gate_percentile   = 0.99,
    ar1_alpha               = None,
    use_steady_state_gain   = True,
    bias_ema_lambda         = 0.995,
    clip_to_bounds          = True,
    cluster_filter          = None,
    save_plots              = True,   # saves PNG, suppresses interactive window
)

In [3]:
# --- Discover all sessions ---
event_paths = get_file_paths(
    f"{base_path}/{experiment}/events/",
    'csv', '', session_type='',
    print_paths=True, print_n=3
)

mouse_ids   = [p.parts[-3] for p in event_paths]
session_ids = [p.parts[-2] for p in event_paths]

print(f"\nFound {len(event_paths)} sessions total, "
      f"{len(skip_indices)} will be skipped.")

Found 72 .csv files
0 d:\data\clickbait-motivate\events\7010\m1\events.csv
1 d:\data\clickbait-motivate\events\7010\m2\events.csv
2 d:\data\clickbait-motivate\events\7010\m4\events.csv

Found 72 sessions total, 1 will be skipped.


In [4]:
# --- Position column selection ---
# Newer SLEAP sessions expose nose_x/nose_y; Bonsai tracking uses
# bonsai_centroid_x/y.  We probe the events DataFrame and fall back
# through the priority list so the batch runs without manual overrides.

from sldata import SessionData

POSITION_PRIORITY = [
    ('bonsai_centroid_x', 'bonsai_centroid_y'),
    ('nose_x',            'nose_y'),
    ('centroid_x',        'centroid_y'),
]

def detect_position_cols(session):
    """Return the first (x_col, y_col) pair that exists in session.events."""
    cols = set(session.events.columns)
    for x_col, y_col in POSITION_PRIORITY:
        if x_col in cols and y_col in cols:
            return x_col, y_col
    raise ValueError(
        f"No known position columns found. Available: {sorted(cols)}"
    )

In [5]:
# --- Batch run ---
all_metrics = []

for ii in range(len(mouse_ids)):
    if ii in skip_indices:
        print(f"[{ii:>3}] SKIP  {mouse_ids[ii]}_{session_ids[ii]}")
        continue

    mouse_id   = mouse_ids[ii]
    session_id = session_ids[ii]
    print(f"\n[{ii:>3}] ── {mouse_id}_{session_id} ──────────────────────")

    try:
        # Load once to detect position columns, then pass them to run_kalman_filter
        # (run_kalman_filter will reload internally, but we need the columns first)
        _session = SessionData(
            mouse_id=mouse_id, session_id=session_id,
            experiment=experiment, base_path=base_path, verbose=False
        )
        x_col, y_col = detect_position_cols(_session)
        del _session

        decoder, metrics = run_kalman_filter(
            mouse_id=mouse_id,
            session_id=session_id,
            experiment=experiment,
            base_path=base_path,
            x_col=x_col,
            y_col=y_col,
            **decoder_params,
        )

        metrics.update(mouse_id=mouse_id, session_id=session_id,
                       x_col=x_col, y_col=y_col)
        all_metrics.append(metrics)
        print(f"    corr_x={metrics['corr_x']:.3f}  corr_y={metrics['corr_y']:.3f}  "
              f"err={metrics['mean_position_error']:.1f} px")

    except Exception as e:
        print(f"    ERROR: {e}")
        all_metrics.append(dict(mouse_id=mouse_id, session_id=session_id, error=str(e)))

print(f"\nBatch complete — {len(all_metrics)} sessions processed.")


[  0] ── 7010_m1 ──────────────────────
Video properties: 888x1968, 30.0 FPS, 81299 frames
KALMAN FILTER POSITION DECODING

1. Loading session 7010_m1
Successfully loaded spike_times from d:\data\clickbait-motivate\kilosorted\7010\m1\spike_times.npy
Successfully loaded spike_templates from d:\data\clickbait-motivate\kilosorted\7010\m1\spike_templates.npy
Successfully loaded templates from d:\data\clickbait-motivate\kilosorted\7010\m1\templates.npy
Successfully loaded sniff from d:\data\clickbait-motivate\preprocessed\7010\m1\sniff.npy
Loaded data: ['spike_times', 'spike_templates', 'templates', 'sniff']
Successfully loaded events from d:\data\clickbait-motivate\events\7010\m1\events.csv
Video properties: 888x1968, 30.0 FPS, 81299 frames
Loaded cluster labels: 14 good, 12 mua, 15 noise
Excluded 15 noise clusters
Found 21290 peaks in sniff signal
Peak times range: 0.8 - 90811.6 ms
Loaded 26 clusters for 7010_m1
Loaded sniff data: 2724400 samples
Found 21290 sniff events

2. Binning neur

In [6]:
# --- Summary table ---
import pandas as pd

df = pd.DataFrame(all_metrics)
cols = ['mouse_id', 'session_id', 'corr_x', 'corr_y', 'r2_x', 'r2_y',
        'mean_position_error', 'linear_corr_x', 'linear_corr_y',
        'p_x', 'p_y', 'p_euclid', 'error']
df = df.reindex(columns=[c for c in cols if c in df.columns])
df

,mouse_id,session_id,corr_x,corr_y,r2_x,r2_y,mean_position_error,linear_corr_x,linear_corr_y,p_x,p_y,p_euclid,error
0,7010,m1,0.057668,-0.017014,-0.253718,-0.350074,603.935010,-0.000758,0.007298,0.180,0.574,0.460,NaN
1,7010,m2,0.064456,0.236725,-0.246170,-0.122529,550.117458,0.008750,0.036228,0.169,0.007,0.002,NaN
2,7010,m4,0.105241,0.026837,-0.108058,-0.165420,626.712704,0.013768,-0.009749,0.045,0.417,0.272,NaN
3,7010,m5,0.061008,0.170888,-0.582694,-0.661956,628.595908,0.040576,0.033792,0.197,0.020,0.014,NaN
4,7010,m6,0.144083,0.116021,-0.154674,-0.563727,651.576635,0.035399,0.046416,0.000,0.110,0.095,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,7013,m15,0.024686,0.058587,-0.304736,-0.254169,642.899255,0.021533,0.011212,0.349,0.224,0.262,NaN
67,7013,m16,0.067643,0.065220,-0.214006,-0.420587,728.242903,0.025038,-0.002223,0.070,0.202,0.212,NaN
68,7013,m17,0.091868,0.020985,-0.259937,-0.212558,611.012056,0.063013,-0.013642,0.044,0.422,0.263,NaN
69,7013,m18,0.123547,0.239908,-0.318101,-0.055744,636.510414,0.044121,0.084673,0.022,0.014,0.007,NaN
